# OpsLens — Dataset-Wide Anomaly Detection Training

**Purpose:** discover every compatible metric CSV in the Microsoft Cloud Monitoring dataset, train an independent Isolation Forest model for each time series, evaluate it on a chronological holdout, and export model artifacts for offline inference.

### Why one model per CSV?
The source dataset contains independent time series. A latency series and a request-rate series have different units and statistical distributions. Training one model per file avoids treating unrelated metrics as if they were one homogeneous signal.

### Notebook workflow
1. Configure paths and training parameters.
2. Import dependencies and shared feature engineering.
3. Discover and inspect all CSV files.
4. Parse and clean compatible time series.
5. Build causal features without using labels.
6. Train/evaluate one model per series.
7. Save artifacts, predictions, and a manifest.
8. Review training coverage and evaluation results.

Run cells from top to bottom. The notebook records skipped files and reasons rather than silently ignoring them.

## 1. Configuration

Update `DATA_DIR` to the root folder containing the extracted dataset. Preserve the dataset's nested folder structure. `OUTPUT_DIR` stores the models and evaluation outputs.

The default settings are a reasonable starting point, not guaranteed optimal values. In particular, `CONTAMINATION` controls the expected proportion of anomalies the Isolation Forest predicts during training.

In [ ]:
from pathlib import Path
import json, hashlib, logging, re, sys
from datetime import datetime, timezone

DATA_DIR = Path("./data").resolve()
OUTPUT_DIR = Path("./artifacts").resolve()

CONTAMINATION = 0.05
TEST_SIZE = 0.30
ROLLING_WINDOW = 30
N_ESTIMATORS = 200
RANDOM_STATE = 42

# Set to a small integer (e.g. 2) for a quick smoke test; use None for every CSV.
MAX_FILES = None

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("opslens-notebook")

assert 0 < CONTAMINATION <= 0.5
assert 0.1 <= TEST_SIZE <= 0.5
assert ROLLING_WINDOW >= 2
assert N_ESTIMATORS >= 50
assert DATA_DIR.exists(), f"Dataset folder not found: {DATA_DIR}"
print("Dataset:", DATA_DIR)
print("Output:", OUTPUT_DIR)

## 2. Imports and shared feature engineering

The `features.py` module is intentionally shared with `predict.py`. It ensures that the exact same feature names and transformations are used during training and inference.

Features include normalized value, rolling mean and spread, deviation from recent behaviour, point-to-point change, and daily time-of-day cycles. The rolling features are causal: they do not use future observations.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Ensure the notebook can import the adjacent shared module.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from features import make_features, FEATURE_NAMES

print("Libraries loaded.")
print("Feature schema:", FEATURE_NAMES)

## 3. Discover the dataset files

This cell recursively finds CSV files, including files nested inside metric-specific directories. It creates an inventory so you can inspect the scope of the run before training.

A CSV is considered *potentially compatible* at this stage. Actual column validation happens during loading.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"DATA_DIR is not a directory: {DATA_DIR}")

csv_files = sorted(DATA_DIR.rglob("*.csv"))
if MAX_FILES is not None:
    csv_files = csv_files[:MAX_FILES]

inventory = []
for path in csv_files:
    try:
        preview = pd.read_csv(path, nrows=3)
        inventory.append({
            "source_file": path.relative_to(DATA_DIR).as_posix(),
            "columns": list(preview.columns),
            "preview_rows": len(preview),
            "readable": True,
        })
    except Exception as exc:
        inventory.append({
            "source_file": path.relative_to(DATA_DIR).as_posix(),
            "columns": [],
            "preview_rows": 0,
            "readable": False,
            "error": str(exc),
        })

inventory_df = pd.DataFrame(inventory)
print(f"Discovered {len(csv_files)} CSV files.")
display(inventory_df.head(20))

## 4. Column detection and data cleaning

The source files are expected to have a timestamp column and a numeric value column. A label column is optional. Column matching is case-insensitive and ignores underscores/spaces, so `TimeStamp` and `timestamp` are both recognized.

Cleaning steps:
- parse timestamps as UTC;
- convert metric values to numeric;
- discard rows with invalid timestamps or values;
- sort chronologically;
- remove duplicate timestamps, keeping the last row.

Labels are converted to binary anomaly flags where possible. Unknown label values remain missing and are excluded from evaluation.

In [ ]:
TIMESTAMP_CANDIDATES = ["timestamp", "time_stamp", "datetime", "date", "time", "ds"]
VALUE_CANDIDATES = ["value", "metric_value", "metricvalue", "measurement", "y"]
LABEL_CANDIDATES = ["label", "is_anomaly", "anomaly", "is_outlier", "target"]

def norm_col(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

def find_column(columns, candidates, required=True):
    lookup = {norm_col(c): c for c in columns}
    for candidate in candidates:
        key = norm_col(candidate)
        if key in lookup:
            return lookup[key]
    if required:
        raise ValueError(f"Missing expected column. Tried {candidates}; found {list(columns)}")
    return None

def parse_labels(series):
    if series is None:
        return None
    true_values = {"1", "true", "yes", "y", "anomaly", "anomalous", "outlier"}
    false_values = {"0", "false", "no", "n", "normal", "ok", ""}
    result = []
    for value in series:
        if pd.isna(value):
            result.append(np.nan)
        elif isinstance(value, (bool, np.bool_)):
            result.append(int(value))
        elif isinstance(value, (int, float, np.integer, np.floating)):
            result.append(int(float(value) != 0))
        else:
            label = str(value).strip().lower()
            result.append(1 if label in true_values else 0 if label in false_values else np.nan)
    return pd.Series(result, dtype="float64")

def load_series(csv_path):
    last_error = None
    for encoding in ("utf-8-sig", "utf-8", "latin-1"):
        try:
            raw = pd.read_csv(csv_path, encoding=encoding)
            break
        except Exception as exc:
            last_error = exc
    else:
        raise ValueError(f"Could not read CSV: {last_error}")

    ts_col = find_column(raw.columns, TIMESTAMP_CANDIDATES)
    value_col = find_column(raw.columns, VALUE_CANDIDATES)
    label_col = find_column(raw.columns, LABEL_CANDIDATES, required=False)

    clean = pd.DataFrame({
        "timestamp": pd.to_datetime(raw[ts_col], errors="coerce", utc=True),
        "value": pd.to_numeric(raw[value_col], errors="coerce"),
        "label": parse_labels(raw[label_col]) if label_col else np.nan,
    })
    clean = clean.replace([np.inf, -np.inf], np.nan).dropna(subset=["timestamp", "value"])
    clean = clean.sort_values("timestamp").drop_duplicates("timestamp", keep="last").reset_index(drop=True)
    if len(clean) < 30:
        raise ValueError(f"Only {len(clean)} usable rows; at least 30 are required.")
    return clean, {"timestamp_column": ts_col, "value_column": value_col, "label_column": label_col}

print("Column detection and cleaning functions are ready.")

## 5. Model identifiers and evaluation helper

Each CSV gets a stable model ID based on its relative path plus a short hash. This avoids collisions when two folders contain files with the same filename.

Evaluation reports precision, recall, F1, and a confusion matrix when usable labels exist in the holdout. If the holdout contains only one class, the notebook still reports the metrics but adds a note because the result is less informative.

In [ ]:
def stable_model_id(relative_path):
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", Path(relative_path).with_suffix("").as_posix()).strip("-").lower()
    slug = slug[-55:] if slug else "series"
    digest = hashlib.sha1(relative_path.encode("utf-8")).hexdigest()[:8]
    return f"{slug}-{digest}"

def evaluate_predictions(y_true, y_pred):
    valid = pd.notna(y_true)
    if not valid.any():
        return {"evaluated_rows": 0, "note": "No usable labels in chronological test partition."}
    yt = np.asarray(y_true[valid], dtype=int)
    yp = np.asarray(y_pred[valid], dtype=int)
    metrics = {
        "evaluated_rows": int(len(yt)),
        "actual_anomalies": int(yt.sum()),
        "predicted_anomalies": int(yp.sum()),
        "precision": float(precision_score(yt, yp, zero_division=0)),
        "recall": float(recall_score(yt, yp, zero_division=0)),
        "f1": float(f1_score(yt, yp, zero_division=0)),
        "confusion_matrix_tn_fp_fn_tp": confusion_matrix(yt, yp, labels=[0, 1]).tolist(),
    }
    if len(np.unique(yt)) < 2:
        metrics["note"] = "Holdout labels contain only one class; interpret metrics cautiously."
    return metrics

print("Model ID and evaluation helpers are ready.")

## 6. Train one model for one time series

The chronological split reserves the last `TEST_SIZE` fraction for evaluation. Normalization mean and standard deviation are calculated from the training partition only, then reused for both the test partition and future inference.

The `StandardScaler` and `IsolationForest` are fitted only on training rows. The model predicts `-1` for anomalies and `1` for normal observations. We convert that to a convenient `1 = anomaly`, `0 = normal` convention in the evaluation output.

The artifact stores:
- fitted Isolation Forest and scaler;
- training-only normalization statistics;
- feature configuration;
- source filename and column mapping;
- holdout evaluation metrics;
- a recent context tail to initialize inference rolling features.

In [ ]:
def train_one_series(csv_path):
    relative = csv_path.relative_to(DATA_DIR).as_posix()
    model_id = stable_model_id(relative)
    series, columns = load_series(csv_path)

    split_idx = int(len(series) * (1.0 - TEST_SIZE))
    split_idx = min(max(split_idx, 20), len(series) - 5)

    train_values = series["value"].iloc[:split_idx].to_numpy(dtype=float)
    reference_mean = float(np.mean(train_values))
    reference_std = float(np.std(train_values))
    if not np.isfinite(reference_std) or reference_std < 1e-12:
        reference_std = 1.0

    all_features = make_features(
        series["value"], series["timestamp"],
        reference_mean, reference_std, ROLLING_WINDOW
    )
    x_train = all_features.iloc[:split_idx].to_numpy()
    x_test = all_features.iloc[split_idx:].to_numpy()

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    model = IsolationForest(
        n_estimators=N_ESTIMATORS,
        contamination=CONTAMINATION,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(x_train_scaled)

    predicted = (model.predict(x_test_scaled) == -1).astype(int)
    decision = model.decision_function(x_test_scaled)
    anomaly_score = -decision

    y_test = series["label"].iloc[split_idx:].reset_index(drop=True)
    metrics = evaluate_predictions(y_test, predicted)

    context_size = max(ROLLING_WINDOW + 2, 10)
    context = series.iloc[max(0, split_idx-context_size):split_idx]
    artifact = {
        "model": model,
        "scaler": scaler,
        "metadata": {
            "model_id": model_id,
            "source_file": relative,
            "rows_total": int(len(series)),
            "train_rows": int(split_idx),
            "test_rows": int(len(series)-split_idx),
            "reference_mean": reference_mean,
            "reference_std": reference_std,
            "rolling_window": int(ROLLING_WINDOW),
            "feature_names": FEATURE_NAMES,
            "columns": columns,
            "contamination": float(CONTAMINATION),
            "n_estimators": int(N_ESTIMATORS),
            "random_state": int(RANDOM_STATE),
            "trained_at_utc": datetime.now(timezone.utc).isoformat(),
            "context": [
                {"timestamp": t.isoformat(), "value": float(v)}
                for t, v in zip(context["timestamp"], context["value"])
            ],
            "evaluation": metrics,
        },
    }

    model_dir = OUTPUT_DIR / "models"
    model_dir.mkdir(parents=True, exist_ok=True)
    artifact_path = model_dir / f"{model_id}.joblib"
    joblib.dump(artifact, artifact_path, compress=3)

    evaluation_dir = OUTPUT_DIR / "evaluation"
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "timestamp": series["timestamp"].iloc[split_idx:].astype(str).to_numpy(),
        "actual_label": y_test.to_numpy(),
        "predicted_anomaly": predicted,
        "anomaly_score_higher_is_more_anomalous": anomaly_score,
    }).to_csv(evaluation_dir / f"{model_id}_test_predictions.csv", index=False)

    return {
        "model_id": model_id,
        "source_file": relative,
        "artifact": f"models/{model_id}.joblib",
        "rows": int(len(series)),
        "train_rows": int(split_idx),
        "test_rows": int(len(series)-split_idx),
        "evaluation": metrics,
    }

print("Per-series training function is ready.")

## 7. Train across every discovered CSV

This is the main training loop. Every discovered CSV is attempted. Incompatible or corrupted files are added to `skipped_files` with an explicit reason; the notebook continues so one bad file does not stop the entire dataset run.

Review the final trained/skipped counts. If many files are skipped, inspect their reasons and add column aliases if the dataset uses a different schema.

In [ ]:
trained_models = []
skipped_files = []

for position, csv_path in enumerate(csv_files, start=1):
    relative = csv_path.relative_to(DATA_DIR).as_posix()
    print(f"[{position}/{len(csv_files)}] {relative}")
    try:
        result = train_one_series(csv_path)
        trained_models.append(result)
        print(f"  trained: {result['model_id']} | rows={result['rows']} | F1={result['evaluation'].get('f1', 'n/a')}")
    except Exception as exc:
        skipped_files.append({"source_file": relative, "reason": str(exc)})
        print(f"  skipped: {exc}")

print(f"\nTraining complete: {len(trained_models)} trained, {len(skipped_files)} skipped.")

## 8. Export manifest and training summary

The manifest is the model catalog used by `predict.py`. It maps each stable model ID to its artifact path and records evaluation summaries. The prediction runner uses this manifest to select the model requested in the JSON file.

Do not manually rename model artifacts without updating the manifest.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "project": "OpsLens",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_root_name": DATA_DIR.name,
    "discovered_csv_files": len(csv_files),
    "trained_models": len(trained_models),
    "skipped_files": len(skipped_files),
    "models": trained_models,
    "skipped": skipped_files,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
(OUTPUT_DIR / "training_summary.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", OUTPUT_DIR / "manifest.json")

## 9. Review model coverage and performance

This summary helps identify which time series were successfully trained and whether the model is finding labelled anomalies. Precision/recall/F1 are meaningful only when the test segment contains suitable labels and enough anomaly examples.

Do not rely on accuracy alone for anomaly detection: anomalies are often rare, so a model that predicts everything as normal can still have high accuracy.

In [ ]:
if trained_models:
    results_df = pd.DataFrame([
        {
            "model_id": m["model_id"],
            "source_file": m["source_file"],
            "rows": m["rows"],
            "train_rows": m["train_rows"],
            "test_rows": m["test_rows"],
            "precision": m["evaluation"].get("precision"),
            "recall": m["evaluation"].get("recall"),
            "f1": m["evaluation"].get("f1"),
            "actual_anomalies": m["evaluation"].get("actual_anomalies"),
            "predicted_anomalies": m["evaluation"].get("predicted_anomalies"),
        }
        for m in trained_models
    ])
    display(results_df)
else:
    print("No models were trained. Inspect skipped_files and the dataset schema.")

skipped_df = pd.DataFrame(skipped_files)
if not skipped_df.empty:
    print("Skipped CSV files:")
    display(skipped_df)

## 10. How to use the exported models with the JSON predictor

1. Open `artifacts/manifest.json`.
2. Copy an exact `model_id`.
3. Paste it into `sample_request.json`.
4. Put new timestamp/value pairs in `observations`.
5. Optionally add recent points in `history`; if omitted, the predictor uses the context tail stored in the artifact.
6. Run `python predict.py --input sample_request.json --artifacts-dir ./artifacts`.

Example shell command:

```bash
python predict.py --input sample_request.json --output prediction_result.json --artifacts-dir ./artifacts
```

The output includes an `anomaly` boolean and an `anomaly_score`. The score is **not a probability**; higher values indicate more anomalous observations according to the model.

### Important limitations
- Each model expects the same metric type and units as its training CSV.
- This is a univariate baseline. It does not yet fuse multiple synchronized metrics.
- A historical context tail from the training period is only a starting point. For realistic live-like tests, pass recent history from the same metric stream.
- Only load trusted `.joblib` artifacts because they use Python pickle serialization.
- Training every CSV increases coverage, not necessarily accuracy. Inspect evaluation metrics and tune contamination using the domain context.